In [ ]:
# CELL 1 — DEPENDÊNCIAS
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'encodec', '-q'])
print('✅ Dependências instaladas!')

In [ ]:
# CELL 2 — SETUP
# ============================================================
import sys, os, math, time, glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datetime import datetime

sys.path.insert(0, '/kaggle/input/datasets/destro01400/nexus-audio-code-beta/')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
n_gpus = torch.cuda.device_count()
print(f'GPUs: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print('✅ Setup OK!')

In [ ]:
# CELL 3 — CONFIGURAÇÃO Beta v1.0 (MS-SSM)
# ============================================================

# ⚙️ SESSÃO: 1 = do zero | 2+ = continua do checkpoint
SESSAO = 5

# Checkpoint pra Sessão 2+ (preenche após salvar o dataset da S1)
CHECKPOINT_RESUME = '/kaggle/input/models/destro01400/nexus-audio/pytorch/beta-1/3/checkpoints/best_session3.pt'  # ex: '/kaggle/input/.../best_session1.pt'

CONFIG = {
    'model': {
        'd_model':    512,
        'n_layers':   4,           # 4 layers MS-SSM (cada um = 3 SSMs em paralelo)
        'ssm_scales': [16, 32, 64],  # 🆕 Beta: escala fina / média / grossa
        'dropout':    0.1,         # 🆕 anti-overfitting no Scale Mixer
        'd_conv':     4,
        'expand':     1,
        'max_seq_len': 750,
        'sample_rate': 24000,
        'encodec_bandwidth': 6.0,
        'use_biofeedback': False,
    },
    'training': {
        'steps':          10000,
        'batch_size':     4,       # 2 por GPU no DataParallel
        'grad_accum':     4,       # batch efetivo = 16
        'lr':             8e-5,
        'warmup_steps':   300,    # S5: warmup médio pra se adaptar aos novos CB_WEIGHTS
        'min_lr':         1e-6,
        'clip_grad':      1.0,
        'checkpoint_every': 500,
        'log_every':      10,
    },
    'data': {
        'tokens_path': '/kaggle/input/datasets/destro01400/nexus-fma-tokens/tokens',
    }
}

print('✅ Config Beta v1.0 carregada!')
print(f'   d_model: {CONFIG["model"]["d_model"]} | scales: {CONFIG["model"]["ssm_scales"]}')
print(f'   n_layers: {CONFIG["model"]["n_layers"]} | dropout: {CONFIG["model"]["dropout"]}')
print(f'   lr: {CONFIG["training"]["lr"]} | warmup: {CONFIG["training"]["warmup_steps"]} steps')
print(f'   batch: {CONFIG["training"]["batch_size"]} | accum: {CONFIG["training"]["grad_accum"]} | efetivo: {CONFIG["training"]["batch_size"] * CONFIG["training"]["grad_accum"]}')
print(f'   steps: {CONFIG["training"]["steps"]} | Sessão: {SESSAO}')


In [ ]:
# CELL 4 — CARREGAR DADOS
# ============================================================

class FastDataset(Dataset):
    def __init__(self, token_files, seq_len=750):
        self.files = token_files
        self.seq_len = seq_len

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            tokens = torch.load(self.files[idx], weights_only=True)
            if tokens.dim() == 3:
                tokens = tokens.squeeze(0)
            if tokens.shape[-1] >= self.seq_len:
                tokens = tokens[:, :self.seq_len]
            else:
                pad = torch.zeros(8, self.seq_len - tokens.shape[-1], dtype=tokens.dtype)
                tokens = torch.cat([tokens, pad], dim=-1)
            return tokens
        except Exception:
            # ✅ Arquivo corrompido — retorna zeros e segue em frente
            return torch.zeros(8, self.seq_len, dtype=torch.long)

token_files = glob.glob(os.path.join(CONFIG['data']['tokens_path'], '*.tokens.pt'))
print(f'✅ Tokens encontrados: {len(token_files)}')

if len(token_files) == 0:
    raise RuntimeError(f'❌ Nenhum token em: {CONFIG["data"]["tokens_path"]}')

dataset = FastDataset(token_files, seq_len=CONFIG['model']['max_seq_len'])
print(f'   FastDataset: {len(dataset)} arquivos | seq_len={CONFIG["model"]["max_seq_len"]}')

loader = DataLoader(
    dataset,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)
print(f'✅ Batches por época: {len(loader)}')

In [ ]:
# CELL 5 — MODELO + OPTIMIZER + SCHEDULER (Beta MS-SSM)
# ============================================================
from src.model.simba_therapeutic import SiMBATherapeutic

cfg  = CONFIG['model']
tcfg = CONFIG['training']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Criando modelo Beta v1.0 (MS-SSM)...')
model = SiMBATherapeutic(
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    ssm_scales=cfg['ssm_scales'],   # 🆕 3 escalas em paralelo
    dropout=cfg['dropout'],          # 🆕 anti-overfitting
    d_conv=cfg['d_conv'],
    expand=cfg['expand'],
    max_seq_len=cfg['max_seq_len'],
    sample_rate=cfg['sample_rate'],
    encodec_bandwidth=cfg['encodec_bandwidth'],
    use_biofeedback=cfg['use_biofeedback'],
)

if torch.cuda.device_count() > 1:
    print(f'   Usando DataParallel com {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)
model = model.to(device)

params = sum(p.numel() for p in model.parameters())
print(f'✅ Parâmetros: {params:,}')

# Optimizer
decay_params    = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = torch.optim.AdamW([
    {'params': decay_params,    'weight_decay': 1e-4},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=tcfg['lr'], betas=(0.9, 0.95))

# Scheduler cosine com warmup (fresco por sessão)
def get_lr(step_in_session):
    min_ratio = tcfg['min_lr'] / tcfg['lr']
    if step_in_session < tcfg['warmup_steps']:
        return min_ratio + (1.0 - min_ratio) * step_in_session / tcfg['warmup_steps']
    progress = (step_in_session - tcfg['warmup_steps']) / max(1, tcfg['steps'] - tcfg['warmup_steps'])
    progress = min(progress, 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_ratio + (1.0 - min_ratio) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

start_step = 0
best_loss  = float('inf')

# Resume (Sessão 2+)
if SESSAO >= 2 and CHECKPOINT_RESUME is not None:
    print(f'\n🔄 Carregando: {CHECKPOINT_RESUME}')
    ckpt = torch.load(CHECKPOINT_RESUME, map_location=device)

    raw = model.module if hasattr(model, 'module') else model
    raw.load_state_dict(ckpt['model_state_dict'], strict=True)

    optimizer.load_state_dict(ckpt['optimizer_state_dict'])

    start_step = ckpt.get('step', 0)
    best_loss  = ckpt.get('best_loss', float('inf'))

    print(f'   ✅ Weights carregados (strict)')
    print(f'   ✅ Optimizer restaurado')
    print(f'   ⚡ Scheduler resetado (cosine fresco pra esta sessão)')
    print(f'   Continuando do step {start_step} | Best loss: {best_loss:.4f}')
else:
    print('ℹ️  Treinando do zero (Sessão 1 — Beta MS-SSM)')

print('\n✅ Tudo pronto!')


In [ ]:
# PATCH — CB_WEIGHTS rebalanceados S5 (Sessão 5)
# CB1 menos dominante, CB5-CB8 com mais voz → mais musicalidade
# ============================================================
import torch
raw = model.module if hasattr(model, 'module') else model
raw.cb_loss_weights = torch.tensor(
    [2.0, 1.5, 1.3, 1.1, 1.0, 0.9, 0.8, 0.7],
    device=raw.cb_loss_weights.device
)
print(f'✅ CB_WEIGHTS S5: {raw.cb_loss_weights.tolist()}')
print('   CB1 menos dominante → modelo cuida mais da textura sonora! 🎵')


In [ ]:
# CELL 6 — LOOP DE TREINO v4.2
# ============================================================
import warnings
warnings.filterwarnings('ignore')

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

tcfg         = CONFIG['training']
total_steps  = start_step + tcfg['steps']
session_name = f'session{SESSAO}'

def ts():
    return datetime.now().strftime('%H:%M:%S')

print('=' * 55)
print(f'[{ts()}] ⚡ TREINO TURBO Beta v1.0 (MS-SSM) — Sessão {SESSAO}')
print(f'[{ts()}] Steps: {start_step}→{total_steps} | lr: {tcfg["lr"]} | warmup: {tcfg["warmup_steps"]}')
print(f'[{ts()}] Batch: {tcfg["batch_size"]} | Accum: {tcfg["grad_accum"]} | Seq: {CONFIG["model"]["max_seq_len"]}')
print('=' * 55)

model.train()
optimizer.zero_grad()

step           = start_step
step_in_session = 0          # ✅ contador local pra scheduler
accum_loss     = 0.0
accum_count    = 0
t_start        = time.time()
t_step_start   = time.time()
speeds         = []
epoch          = 0

while step < total_steps:
    epoch += 1
    print(f'[{ts()}] 📍 Época {epoch}')

    for batch in loader:
        if step >= total_steps:
            break

        tokens = batch.to(device)

        output = model(tokens, labels=tokens)

        if isinstance(output, dict):
            loss = output.get('loss')
            if loss is None:
                logits = output['logits']        # [B, 8, T, 1024]
                B, C, T, V = logits.shape
                pred = logits[:, :, :-1, :].reshape(-1, V)
                tgt  = tokens[:, :, 1:].reshape(-1)
                loss = nn.CrossEntropyLoss()(pred, tgt)
        else:
            loss = output

        if isinstance(loss, torch.Tensor) and loss.dim() > 0:
            loss = loss.mean()

        loss = loss / tcfg['grad_accum']
        loss.backward()

        accum_loss  += loss.item() * tcfg['grad_accum']
        accum_count += 1

        if accum_count % tcfg['grad_accum'] == 0:

            # Diagnóstico de gradiente (primeiros 500 steps da sessão)
            if step_in_session < 500 and step_in_session % 50 == 0:
                total_norm = sum(p.grad.norm().item() ** 2
                                 for p in model.parameters() if p.grad is not None) ** 0.5
                zero_grads = sum(1 for p in model.parameters() if p.grad is None)
                status = '✅ normal' if 0.1 < total_norm < 50 else '⚠️  ANORMAL'
                print(f'   🔍 Grad norm: {total_norm:.4f} {status} | Params sem grad: {zero_grads}')

            nn.utils.clip_grad_norm_(model.parameters(), tcfg['clip_grad'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            step            += 1
            step_in_session += 1
            current_loss     = accum_loss / tcfg['grad_accum']
            accum_loss       = 0.0

            elapsed = time.time() - t_step_start
            spm = 60.0 / elapsed if elapsed > 0 else 0  # steps por minuto real
            speeds.append(spm)
            if len(speeds) > 20:
                speeds.pop(0)
            avg_spm   = sum(speeds) / len(speeds)
            remaining = (total_steps - step) / avg_spm if avg_spm > 0 else 0  # ETA em minutos
            t_step_start = time.time()

            current_lr = optimizer.param_groups[0]['lr']

            if step % tcfg['log_every'] == 0:
                perplexity = torch.exp(torch.tensor(current_loss)).item()
                print(f'[{ts()}] Step {step}/{total_steps} | Loss: {current_loss:.4f} | Perplexidade: {perplexity:.1f} | '
                      f'lr: {current_lr:.2e} | {avg_spm:.1f} spm | ETA: {remaining:.0f}min')

            # Salva best
            if current_loss < best_loss:
                best_loss = current_loss
                raw = model.module if hasattr(model, 'module') else model
                torch.save({
                    'model_state_dict':     raw.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'step': step,
                    'best_loss': best_loss,
                    'session': SESSAO,
                    'config': CONFIG,
                }, f'/kaggle/working/checkpoints/best_{session_name}.pt')

            # Checkpoint periódico
            if step % tcfg['checkpoint_every'] == 0:
                raw = model.module if hasattr(model, 'module') else model
                torch.save({
                    'model_state_dict':     raw.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'step': step,
                    'best_loss': best_loss,
                    'session': SESSAO,
                    'config': CONFIG,
                }, f'/kaggle/working/checkpoints/step{step}_{session_name}.pt')
                print(f'[{ts()}] 💾 Checkpoint step{step} | Best loss: {best_loss:.4f}')

    else:
        continue
    break

# Salva final
raw = model.module if hasattr(model, 'module') else model
final_path = f'/kaggle/working/checkpoints/nexus_beta_v1.0_{session_name}.pt'
torch.save({
    'model_state_dict':     raw.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'step': step,
    'best_loss': best_loss,
    'session': SESSAO,
    'config': CONFIG,
}, final_path)

tempo = (time.time() - t_start) / 60
print('=' * 55)
print(f'[{ts()}] ✅ SESSÃO {SESSAO} COMPLETA!')
print(f'[{ts()}]    Tempo: {tempo:.1f}min | Steps: {start_step}→{step}')
print(f'[{ts()}]    Best loss: {best_loss:.4f}')
print(f'[{ts()}]    Salvo em: {final_path}')
print('=' * 55)
if SESSAO == 1:
    print()
    print('📌 PRÓXIMO PASSO — Sessão 2:')
    print(f'   1. Salva /kaggle/working/checkpoints/ como dataset: nexus-beta-v1-session1')
    print(f'   2. Muda SESSAO = 2')
    print(f'   3. Atualiza CHECKPOINT_RESUME pro best_session1.pt')
    print(f'   4. Roda!')

In [ ]:
# CELL 7 — ARQUIVOS SALVOS
# ============================================================
import subprocess
result = subprocess.run(['ls', '-lh', '/kaggle/working/checkpoints/'], capture_output=True, text=True)
print(result.stdout)
print(f'Steps realizados: {step - start_step}')
print(f'Best loss:        {best_loss:.4f}')
print(f'Sessão:           {SESSAO}')

In [ ]:
# CELL 8 — LIMPEZA DE CHECKPOINTS
# ============================================================
import os, glob

checkpoint_dir = '/kaggle/working/checkpoints'

# ✅ FIX v4.2: ordenação NUMÉRICA (não textual)
# textual: step9500 > step10000 (errado!)
# numérica: step9500 < step10000 (correto!)
step_ckpts = sorted(
    glob.glob(f'{checkpoint_dir}/step*_session*.pt'),
    key=lambda x: int(x.split('step')[1].split('_')[0])
)

ultimo = step_ckpts[-1] if step_ckpts else None

deletados = 0
for ckpt in step_ckpts:
    if ckpt == ultimo:
        continue
    os.remove(ckpt)
    deletados += 1

print(f'🗑️  Deletados: {deletados} checkpoints intermediários')
print(f'✅ Mantidos:')
print(f'   📌 Best:   best_{session_name}.pt')
if ultimo:
    print(f'   📌 Último: {os.path.basename(ultimo)}')

restantes = os.listdir(checkpoint_dir)
total_gb = sum(os.path.getsize(f'{checkpoint_dir}/{f}') for f in restantes) / 1e9
print(f'\n📦 Pasta final: {len(restantes)} arquivos | {total_gb:.2f} GB')

In [ ]:
import subprocess
import requests

TOKEN_DO_BOT = '7781713605:AAFzHiduYdHUMAWuw56MY5VHq4LRbty4hZQ'
SEU_CHAT_ID = '6908956487'

# O texto entre aspas triplas ou parênteses deixa você organizar melhor no Python
# Olha os \n fazendo as quebras e os asteriscos fazendo o negrito!
MENSAGEM = (
    "🚨 *Alerta da SnaX Company* 🚨\n\n"
    "Destro, acorda que tem aqui tem coisa!\n"
    "O treinamento da *Nexus-Audio* acabou de finalizar lá no Kaggle.\n\n"
    "Status:\n"
    f"Loss: *{best_loss:.4f}*\n"
    f"Step: {step}\n"
    f"Sessão: {SESSAO}\n"
    f"Steps realizados: {step - start_step}\n"
    f"Arquivos: {len(restantes)} arquivos | {total_gb:.2f} GB\n"
    f"Parâmetros: {params:,}\n"
    f"Tokens usados: {len(token_files)}"

url = f"https://api.telegram.org/bot{TOKEN_DO_BOT}/sendMessage"

# O truque supremo: a linha "parse_mode": "Markdown" faz o negrito funcionar!
payload = {
    "chat_id": SEU_CHAT_ID, 
    "text": MENSAGEM,
    "parse_mode": "Markdown"
}

resposta = requests.post(url, json=payload)

if resposta.status_code == 200:
    print("Aviso chique enviado pro celular do CEO! 📱✨")
else:
    print("Deu ruim:", resposta.text)
